In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("bat-fwf5j").project("bat-keypoint-hqmj3")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


In [ ]:
import os
import glob
import yaml
from ultralytics import YOLO

# 1. 프로젝트 경로 확정
dataset_root = "/content/Bat-Keypoint-1"

if not os.path.exists(dataset_root):
    # 경로가 다를 경우 검색
    found = glob.glob("/content/Bat-Keypoint*")
    if found:
        dataset_root = found[0]
    else:
        print("🚨 프로젝트 폴더를 찾을 수 없습니다! 데이터셋 다운로드부터 다시 해야 할 수 있습니다.")
        # 진행 불가 시 멈춤
        dataset_root = None

if dataset_root:
    print(f"🔧 작업 대상 폴더: {dataset_root}")

    # ---------------------------------------------------------
    # 2. 방해되는 모든 캐시 파일 제거
    # ---------------------------------------------------------
    # labels.cache 파일이 보이면 무조건 지웁니다.
    cache_list = glob.glob(f"{dataset_root}/**/*.cache", recursive=True)
    for cache in cache_list:
        try:
            os.remove(cache)
            print(f"🗑️ 캐시 삭제: {cache}")
        except:
            pass

    # ---------------------------------------------------------
    # 3. 라벨 파일 직접 뜯어보고 설정값 계산하기
    # ---------------------------------------------------------
    train_labels = glob.glob(f"{dataset_root}/train/labels/*.txt")

    if not train_labels:
        print("🚨 훈련용 라벨 파일(.txt)이 하나도 없습니다! 데이터셋 다운로드가 잘못되었습니다.")
    else:
        # 첫 번째 파일을 열어서 분석
        sample_file = train_labels[0]
        with open(sample_file, 'r') as f:
            lines = f.readlines()

        if not lines:
            print("🚨 라벨 파일이 비어있습니다!")
        else:
            # 첫 줄의 숫자 개수 세기
            first_line_data = lines[0].strip().split()
            col_count = len(first_line_data)
            print(f"🔎 데이터 분석: 한 줄에 숫자가 {col_count}개 있습니다.")
            print(f"   내용 예시: {first_line_data[:10]}...")

            # YOLO 포맷 계산: (전체 - 5개박스정보) / 키포인트차원
            # 보통 2차원(x,y) 이거나 3차원(x,y,visible) 입니다.

            kpts_num = 0
            kpts_dim = 0

            # 경우의 수 계산
            rem = col_count - 5
            if rem > 0:
                if rem % 2 == 0:
                    kpts_num = rem // 2
                    kpts_dim = 2
                    print(f"👉 분석 결과: 키포인트 {kpts_num}개 (x, y 포맷) 유력")
                elif rem % 3 == 0:
                    kpts_num = rem // 3
                    kpts_dim = 3
                    print(f"👉 분석 결과: 키포인트 {kpts_num}개 (x, y, visibility 포맷) 유력")
                else:
                    print("⚠️ 경고: 데이터 개수가 일반적인 키포인트 포맷과 맞지 않습니다.")

            # ---------------------------------------------------------
            # 4. yaml 파일 완벽하게 다시 쓰기
            # ---------------------------------------------------------
            yaml_path = f"{dataset_root}/data.yaml"

            # 내용을 새로 작성합니다 (기존 내용 무시)
            new_data = {
                'path': dataset_root,         # 절대 경로
                'train': 'train/images',      # 상대 경로
                'val': 'valid/images',        # 상대 경로
                'names': {0: 'bat'},          # 클래스 이름
            }

            # 키포인트 정보가 확인되면 추가
            if kpts_num > 0:
                new_data['kpt_shape'] = [kpts_num, kpts_dim]

            with open(yaml_path, 'w') as f:
                yaml.dump(new_data, f)

            print(f"✅ data.yaml 재생성 완료: {new_data}")

            # ---------------------------------------------------------
            # 5. 학습 시작
            # ---------------------------------------------------------
            print("🚀 학습을 시작합니다!")
            model = YOLO('yolov8n-pose.pt')
            # cache=False 옵션은 없지만, 위에서 파일을 지웠으니 효과가 같습니다.
            model.train(data=yaml_path, epochs=50, imgsz=640, plots=True)

🔧 작업 대상 폴더: /content/Bat-Keypoint-1
🗑️ 캐시 삭제: /content/Bat-Keypoint-1/train/labels.cache
🔎 데이터 분석: 한 줄에 숫자가 14개 있습니다.
   내용 예시: ['0', '0.41495531249999995', '0.5454847222222222', '0.24107140624999998', '0.23700111111111113', '0.518526796875', '0.4733970833333333', '2', '0.32299109375', '0.6401618055555556']...
👉 분석 결과: 키포인트 3개 (x, y, visibility 포맷) 유력
✅ data.yaml 재생성 완료: {'path': '/content/Bat-Keypoint-1', 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'bat'}, 'kpt_shape': [3, 3]}
🚀 학습을 시작합니다!
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Bat-Keypoint-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, em